In [4]:


import json
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 1. Prepare tools and settings
np.random.seed(25)

N_SKUS = 500
SERVICE_LEVEL_Z = 1.645
DAYS = 365

# 2. Generate product data
sku_ids = [f"SKU-{i:03d}" for i in range(1, N_SKUS + 1)]

annual_demand = np.random.lognormal(
    mean=6.75, sigma=0.8, size=N_SKUS
).astype(int)
annual_demand = np.maximum(annual_demand, 1)

unit_cost = np.random.lognormal(
    mean=2.95, sigma=0.65, size=N_SKUS
)

lead_time = np.random.randint(5, 31, size=N_SKUS)
demand_cv = np.random.uniform(0.2, 0.8, size=N_SKUS)

df = pd.DataFrame({
    "SKU": sku_ids,
    "Annual_Demand": annual_demand,
    "Unit_Cost": unit_cost,
    "Lead_Time": lead_time,
    "Demand_CV": demand_cv
})

df["Annual_Consumption_Value"] = (
    df["Annual_Demand"] * df["Unit_Cost"]
)

# 3. Rank products by annual consumption value
df = df.sort_values(
    "Annual_Consumption_Value", ascending=False
).reset_index(drop=True)

total_value = df["Annual_Consumption_Value"].sum()

df["Cumulative_Value"] = (
    df["Annual_Consumption_Value"].cumsum()
)

df["Cumulative_Share"] = df["Cumulative_Value"] / total_value

# 4. Classify products using cumulative value shares
df["ABC_Class"] = pd.cut(
    df["Cumulative_Share"],
    bins=[0, 0.80, 0.95, 1.000001],
    labels=["A", "B", "C"],
    include_lowest=True
)

# 5. Summarize ABC classes
summary = df.groupby(
    "ABC_Class", observed=False
).agg(
    SKU_Count=("SKU", "count"),
    Total_Value=("Annual_Consumption_Value", "sum")
)

summary["SKU_Share_Percent"] = (
    summary["SKU_Count"] / N_SKUS * 100
)

summary["Value_Share_Percent"] = (
    summary["Total_Value"] / total_value * 100
)

# 6. Simulate 365 days of demand for Class A items
df["Daily_Mean"] = 0.0
df["Daily_Std"] = 0.0
df["Safety_Stock"] = 0
df["Demand_During_Lead_Time"] = 0
df["Reorder_Point"] = 0

for idx in df.index[df["ABC_Class"] == "A"]:
    daily_mean = df.loc[idx, "Annual_Demand"] / DAYS
    cv = df.loc[idx, "Demand_CV"]

    # Gamma distribution: shape = 1/CV^2
    shape = 1 / (cv ** 2)
    scale = daily_mean / shape

    simulated_demand = np.random.gamma(
        shape, scale, DAYS
    )

    mean_demand = simulated_demand.mean()
    std_demand = simulated_demand.std(ddof=1)

    # 7. Safety stock and reorder point
    lt = df.loc[idx, "Lead_Time"]

    safety_stock = int(np.ceil(
        SERVICE_LEVEL_Z * std_demand * np.sqrt(lt)
    ))

    lead_time_demand = int(np.ceil(mean_demand * lt))

    reorder_point = lead_time_demand + safety_stock

    df.loc[idx, "Daily_Mean"] = mean_demand
    df.loc[idx, "Daily_Std"] = std_demand
    df.loc[idx, "Safety_Stock"] = safety_stock
    df.loc[idx, "Demand_During_Lead_Time"] = lead_time_demand
    df.loc[idx, "Reorder_Point"] = reorder_point

df["Safety_Stock_Value"] = (
    df["Safety_Stock"] * df["Unit_Cost"]
)

class_a = df[df["ABC_Class"] == "A"].copy()

# 8. Create all three charts in one figure
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Chart 1: Pareto chart using 20 groups of 25 SKUs
group_size = N_SKUS // 20

group_values = (
    df["Annual_Consumption_Value"]
    .to_numpy()
    .reshape(20, group_size)
    .sum(axis=1)
)

group_shares = group_values / total_value * 100
cumulative_shares = np.cumsum(group_shares)
groups = np.arange(1, 21)

ax1 = axes[0]
ax1.bar(groups, group_shares, color="cornflowerblue")
ax1.set_xlabel("SKU group (25 SKUs per group)")
ax1.set_ylabel("Share of total value (%)")
ax1.set_title("Pareto Chart")

ax1_line = ax1.twinx()
ax1_line.plot(
    groups, cumulative_shares,
    color="orangered", marker="o", markersize=3
)
ax1_line.set_ylabel("Cumulative share (%)")
ax1_line.set_ylim(0, 105)
ax1_line.axhline(80, linestyle="--", color="gray", linewidth=1)
ax1_line.axhline(95, linestyle=":", color="gray", linewidth=1)

# Chart 2: SKU share versus value share
classes = ["A", "B", "C"]
x = np.arange(len(classes))
width = 0.35

ax2 = axes[1]
sku_shares = summary.loc[classes, "SKU_Share_Percent"]
value_shares = summary.loc[classes, "Value_Share_Percent"]

ax2.bar(x - width / 2, sku_shares, width, label="Share of SKUs")
ax2.bar(x + width / 2, value_shares, width, label="Share of value")

ax2.set_xticks(x)
ax2.set_xticklabels(classes)
ax2.set_ylabel("Share (%)")
ax2.set_title("ABC Classes: SKUs vs Value")
ax2.legend(fontsize=8)

# Chart 3: Reorder points for 12 largest Class A items
top12 = class_a.nlargest(12, "Annual_Consumption_Value")
y = np.arange(len(top12))

ax3 = axes[2]
ax3.barh(
    y, top12["Demand_During_Lead_Time"],
    label="Demand during lead time"
)
ax3.barh(
    y, top12["Safety_Stock"],
    left=top12["Demand_During_Lead_Time"],
    label="Safety stock"
)

ax3.set_yticks(y)
ax3.set_yticklabels(top12["SKU"], fontsize=7)
ax3.invert_yaxis()
ax3.set_xlabel("Units")
ax3.set_title("Reorder Point: Top 12 Class A")
ax3.legend(fontsize=7)

fig.tight_layout()
fig.savefig("chart.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# 9. Save summary metrics in JSON
results = {
    "total_skus": int(len(df)),
    "total_annual_consumption_value": round(float(total_value), 2),
    "class_summary": {},
    "class_a_average_safety_stock": round(
        float(class_a["Safety_Stock"].mean()), 2
    ),
    "class_a_total_safety_stock_units": int(
        class_a["Safety_Stock"].sum()
    ),
    "class_a_safety_stock_inventory_value": round(
        float(class_a["Safety_Stock_Value"].sum()), 2
    ),
    "service_level_percent": 95,
    "z_value": SERVICE_LEVEL_Z
}

for abc_class in classes:
    results["class_summary"][abc_class] = {
        "sku_count": int(summary.loc[abc_class, "SKU_Count"]),
        "sku_share_percent": round(
            float(summary.loc[abc_class, "SKU_Share_Percent"]), 2
        ),
        "total_value": round(
            float(summary.loc[abc_class, "Total_Value"]), 2
        ),
        "value_share_percent": round(
            float(summary.loc[abc_class, "Value_Share_Percent"]), 2
        )
    }

with open("results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4)

# Save the complete product table as an additional useful output
df.to_csv("inventory_abc.csv", index=False)

# Display results
print("\nABC CLASSIFICATION SUMMARY")
print(summary.round(2))

print("\nTOTAL ANNUAL CONSUMPTION VALUE:")
print(f"${total_value:,.2f}")

print("\nCLASS A SAFETY STOCK")
print(f"Average units per SKU: {class_a['Safety_Stock'].mean():.2f}")
print(f"Inventory value tied up: ${class_a['Safety_Stock_Value'].sum():,.2f}")

print("\nFiles created: chart.png, results.json, inventory_abc.csv")



ABC CLASSIFICATION SUMMARY
           SKU_Count  Total_Value  SKU_Share_Percent  Value_Share_Percent
ABC_Class                                                                
A                209  12071693.68               41.8                79.97
B                154   2266157.57               30.8                15.01
C                137    756680.03               27.4                 5.01

TOTAL ANNUAL CONSUMPTION VALUE:
$15,094,531.28

CLASS A SAFETY STOCK
Average units per SKU: 17.79
Inventory value tied up: $107,523.34

Files created: chart.png, results.json, inventory_abc.csv
